Código gerado pedindo exemplos ao Amazon Q sobre integrações entre o OpenCV e a biblioteca Tkinter

In [4]:
import tkinter as tk

In [10]:


# Create the main window
window = tk.Tk()
window.title("Simple GUI Example")
window.geometry("300x200")

# Create a label
label = tk.Label(window, text="Hello, World!",background="red")
label.pack(pady=10)

# Create a button
button = tk.Button(window, text="Click Me!", command=lambda: print("Button clicked!"))
button.pack(pady=50)

# Start the application
window.mainloop()


In [11]:
import tkinter as tk
from tkinter import ttk
import cv2
from PIL import Image, ImageTk

In [ ]:


class CameraApp:
    def __init__(self, window):
        self.window = window
        self.window.title("Camera Viewer")

        # Create a frame to contain the video
        self.video_frame = ttk.Frame(window)
        self.video_frame.pack(padx=10, pady=10)

        # Create label to display the video
        self.video_label = ttk.Label(self.video_frame)
        self.video_label.pack()

        # Initialize video capture
        self.cap = cv2.VideoCapture(0)  # 0 for default camera

        # Buttons frame
        self.button_frame = ttk.Frame(window)
        self.button_frame.pack(pady=5)

        # Start button
        self.start_button = ttk.Button(self.button_frame, text="Start", command=self.start_video)
        self.start_button.pack(side=tk.LEFT, padx=5)

        # Stop button
        self.stop_button = ttk.Button(self.button_frame, text="Stop", command=self.stop_video)
        self.stop_button.pack(side=tk.LEFT, padx=5)

        self.is_running = False

    def start_video(self):
        self.is_running = True
        self.update_frame()

    def stop_video(self):
        self.is_running = False

    def update_frame(self):
        if self.is_running:
            ret, frame = self.cap.read()
            if ret:
                # Convert frame from BGR to RGB
                frame_rgb = cv2.cvtColor(frame, cv2.COLOR_BGR2RGB)
                
                # Convert frame to PhotoImage
                img = Image.fromarray(frame_rgb)
                imgtk = ImageTk.PhotoImage(image=img)
                
                # Update label
                self.video_label.imgtk = imgtk
                self.video_label.configure(image=imgtk)
                
                # Schedule next update
                self.window.after(10, self.update_frame)

    def __del__(self):
        if self.cap.isOpened():
            self.cap.release()

# Create main window and start app
root = tk.Tk()
app = CameraApp(root)
root.mainloop()


In [12]:
import tkinter as tk
from tkinter import ttk
import cv2
from PIL import Image, ImageTk
import numpy as np

In [13]:


class ImageProcessingApp:
    def __init__(self, window):
        self.window = window
        self.window.title("Image Processing App")

        # Main frame
        self.main_frame = ttk.Frame(window)
        self.main_frame.pack(padx=10, pady=10)

        # Video frame
        self.video_frame = ttk.Label(self.main_frame)
        self.video_frame.grid(row=0, column=0, padx=5, pady=5)

        # Controls frame
        self.controls_frame = ttk.Frame(self.main_frame)
        self.controls_frame.grid(row=1, column=0, pady=10)

        # Processing options
        self.process_var = tk.StringVar(value="normal")
        
        ttk.Radiobutton(self.controls_frame, text="Normal", 
                        variable=self.process_var, value="normal").pack(side=tk.LEFT, padx=5)
        ttk.Radiobutton(self.controls_frame, text="Grayscale", 
                        variable=self.process_var, value="gray").pack(side=tk.LEFT, padx=5)
        ttk.Radiobutton(self.controls_frame, text="Edge Detection", 
                        variable=self.process_var, value="edge").pack(side=tk.LEFT, padx=5)
        ttk.Radiobutton(self.controls_frame, text="Blur", 
                        variable=self.process_var, value="blur").pack(side=tk.LEFT, padx=5)

        # Initialize video capture
        self.cap = cv2.VideoCapture(0)
        self.is_running = True
        self.update_frame()

    def process_frame(self, frame):
        mode = self.process_var.get()
        
        if mode == "normal":
            return frame
        elif mode == "gray":
            return cv2.cvtColor(cv2.cvtColor(frame, cv2.COLOR_BGR2GRAY), cv2.COLOR_GRAY2BGR)
        elif mode == "edge":
            gray = cv2.cvtColor(frame, cv2.COLOR_BGR2GRAY)
            edges = cv2.Canny(gray, 100, 200)
            return cv2.cvtColor(edges, cv2.COLOR_GRAY2BGR)
        elif mode == "blur":
            return cv2.GaussianBlur(frame, (15, 15), 0)
        
        return frame

    def update_frame(self):
        if self.is_running:
            ret, frame = self.cap.read()
            if ret:
                # Process the frame
                processed_frame = self.process_frame(frame)
                
                # Convert to RGB for display
                frame_rgb = cv2.cvtColor(processed_frame, cv2.COLOR_BGR2RGB)
                
                # Convert to PhotoImage
                img = Image.fromarray(frame_rgb)
                imgtk = ImageTk.PhotoImage(image=img)
                
                # Update label
                self.video_frame.imgtk = imgtk
                self.video_frame.configure(image=imgtk)
            
            self.window.after(10, self.update_frame)

    def __del__(self):
        if self.cap.isOpened():
            self.cap.release()

# Create and run application
root = tk.Tk()
app = ImageProcessingApp(root)
root.mainloop()


In [2]:
import tkinter as tk
from tkinter import ttk
import cv2
import PIL.Image, PIL.ImageTk
import numpy as np


In [3]:

class DrawingApp:
    def __init__(self, window):
        self.window = window
        self.window.title("Drawing Application")
        
        # Initialize canvas
        self.canvas_width = 600
        self.canvas_height = 600
        
        # Create canvas
        self.canvas = tk.Canvas(window, width=self.canvas_width, height=self.canvas_height)
        self.canvas.pack(side=tk.LEFT, pady=10, padx=10)
        
        # Create blank image for drawing
        self.image = np.ones((self.canvas_height, self.canvas_width, 3), dtype=np.uint8) * 255
        self.temp_image = self.image.copy()
        
        # Drawing parameters
        self.drawing = False
        self.last_x = None
        self.last_y = None
        self.color = (0,0,0)  # Black color
        self.thickness = 5
        
        # Bind mouse events
        self.canvas.bind("<Button-1>", self.start_drawing)
        self.canvas.bind("<B1-Motion>", self.draw)
        self.canvas.bind("<ButtonRelease-1>", self.stop_drawing)
        
        # Create controls
        self.controls_frame = ttk.Frame(window)
        self.controls_frame.pack(side=tk.RIGHT, padx=5)
        
        # Color selection
        ttk.Button(self.controls_frame, text="Black", command=lambda: self.set_color((0,0,0))).pack()
        ttk.Button(self.controls_frame, text="Red", command=lambda: self.set_color((0,0,255))).pack()
        ttk.Button(self.controls_frame, text="Green", command=lambda: self.set_color((0,255,0))).pack()
        ttk.Button(self.controls_frame, text="Blue", command=lambda: self.set_color((255,0,0))).pack()
        
        # Clear button
        ttk.Button(self.controls_frame, text="Clear", command=self.clear_canvas).pack()
        
        self.update_canvas()
        
    def start_drawing(self, event):
        self.drawing = True
        self.last_x = event.x
        self.last_y = event.y
        
    def draw(self, event):
        if self.drawing:
            current_x, current_y = event.x, event.y
            # Draw on OpenCV image
            cv2.line(self.image, 
                     (self.last_x, self.last_y), 
                     (current_x, current_y),
                     self.color,
                     self.thickness)
            self.last_x = current_x
            self.last_y = current_y
            self.update_canvas()
            
    def stop_drawing(self, event):
        self.drawing = False
        
    def set_color(self, color):
        self.color = color
        
    def clear_canvas(self):
        self.image = np.ones((self.canvas_height, self.canvas_width, 3), dtype=np.uint8) * 255
        self.update_canvas()
        
    def update_canvas(self):
        # Convert OpenCV image to PhotoImage
        rgb_image = cv2.cvtColor(self.image, cv2.COLOR_BGR2RGB)
        pil_image = PIL.Image.fromarray(rgb_image)
        self.photo = PIL.ImageTk.PhotoImage(image=pil_image)
        
        # Update canvas
        self.canvas.create_image(0, 0, image=self.photo, anchor=tk.NW)

if __name__ == "__main__":
    root = tk.Tk()
    app = DrawingApp(root)
    root.mainloop()


In [16]:
import tkinter as tk
from tkinter import messagebox
from tkinter import ttk

class LoginWindow:
    def __init__(self):
        self.window = tk.Tk()
        self.window.title("Login")
        self.window.geometry("300x200")
        
        # Center the window contents
        self.frame = ttk.Frame(self.window, padding="20")
        self.frame.grid(row=0, column=0, sticky=(tk.W, tk.E, tk.N, tk.S))
        
        # Username
        ttk.Label(self.frame, text="Username:").grid(row=0, column=0, pady=5)
        self.username = ttk.Entry(self.frame)
        self.username.grid(row=0, column=1, pady=5)
        
        # Password
        ttk.Label(self.frame, text="Password:").grid(row=1, column=0, pady=5)
        self.password = ttk.Entry(self.frame, show="*")
        self.password.grid(row=1, column=1, pady=5)
        
        # Login button
        ttk.Button(self.frame, text="Login", command=self.login).grid(row=2, column=0, columnspan=2, pady=20)
        
        # For this example, we'll use a simple hardcoded credential
        self.valid_username = "admin"
        self.valid_password = "password"
        
    def login(self):
        if (self.username.get() == self.valid_username and 
            self.password.get() == self.valid_password):
            self.window.withdraw()  # Hide login window
            MainApplication(self.window)  # Open main application
        else:
            messagebox.showerror("Error", "Invalid credentials!")
            
    def run(self):
        self.window.mainloop()

class MainApplication:
    def __init__(self, login_window):
        self.login_window = login_window
        self.window = tk.Toplevel()
        self.window.title("Main Application")
        self.window.geometry("600x400")
        
        # Menu Bar
        self.create_menu_bar()
        
        # Main content
        self.create_main_content()
        
        # Handle window close
        self.window.protocol("WM_DELETE_WINDOW", self.on_closing)
        
    def create_menu_bar(self):
        menubar = tk.Menu(self.window)
        self.window.config(menu=menubar)
        
        # File Menu
        file_menu = tk.Menu(menubar, tearoff=0)
        menubar.add_cascade(label="File", menu=file_menu)
        file_menu.add_command(label="New Window", command=self.open_new_window)
        file_menu.add_separator()
        file_menu.add_command(label="Logout", command=self.logout)
        file_menu.add_command(label="Exit", command=self.on_closing)
        
    def create_main_content(self):
        # Main frame
        main_frame = ttk.Frame(self.window, padding="20")
        main_frame.grid(row=0, column=0, sticky=(tk.W, tk.E, tk.N, tk.S))
        
        # Welcome message
        ttk.Label(
            main_frame, 
            text="Welcome to the Main Application!",
            font=("Arial", 16)
        ).grid(row=0, column=0, pady=20)
        
        # Button to open new window
        ttk.Button(
            main_frame, 
            text="Open New Window",
            command=self.open_new_window
        ).grid(row=1, column=0, pady=10)
        
    def open_new_window(self):
        new_window = tk.Toplevel(self.window)
        new_window.title("New Window")
        new_window.geometry("300x200")
        
        # Add some content to the new window
        frame = ttk.Frame(new_window, padding="20")
        frame.grid(row=0, column=0, sticky=(tk.W, tk.E, tk.N, tk.S))
        
        ttk.Label(
            frame, 
            text="This is a new window!",
            font=("Arial", 12)
        ).grid(row=0, column=0, pady=20)
        
        ttk.Button(
            frame, 
            text="Close",
            command=new_window.destroy
        ).grid(row=1, column=0)
        
    def logout(self):
        self.window.destroy()
        self.login_window.deiconify()  # Show login window again
        
    def on_closing(self):
        if messagebox.askokcancel("Quit", "Do you want to quit?"):
            self.window.destroy()
            self.login_window.destroy()

if __name__ == "__main__":
    app = LoginWindow()
    app.run()


In [21]:
import tkinter as tk

class KeyboardControlApp:
    def __init__(self, root):
        self.root = root
        self.root.title("Keyboard Control Example")
        self.root.geometry("400x300")
        
        # Create a canvas where we'll display a movable object
        self.canvas = tk.Canvas(root, width=400, height=200, bg="white")
        self.canvas.pack(pady=20)
        
        # Create a rectangle that will be controlled by keyboard
        self.rect = self.canvas.create_rectangle(180, 80, 220, 120, fill="blue")
        
        # Create a label to show pressed keys
        self.key_label = tk.Label(root, text="Press arrow keys to move the square\nPress 'c' to change color\nPress 'r' to reset position")
        self.key_label.pack(pady=10)
        
        self.status_label = tk.Label(root, text="Status: Ready")
        self.status_label.pack(pady=5)
        
        # Bind keyboard events to the window
        self.root.bind("<Key>", self.key_press)
        
        # Set focus to the window so it can receive keyboard events
        self.root.focus_set()
        
        # Available colors for cycling
        self.colors = ["blue", "red", "green", "orange", "purple"]
        self.current_color = 0

    def key_press(self, event):
        """Handle keyboard events"""
        key = event.keysym
        
        # Update status label
        self.status_label.config(text=f"Status: Key pressed - {key}")
        
        # Move the rectangle based on arrow keys
        if key == "Left":
            self.canvas.move(self.rect, -10, 0)
        elif key == "Right":
            self.canvas.move(self.rect, 10, 0)
        elif key == "Up":
            self.canvas.move(self.rect, 0, -10)
        elif key == "Down":
            self.canvas.move(self.rect, 0, 10)
        # Change color when 'c' is pressed
        elif key.lower() == "c":
            self.current_color = (self.current_color + 1) % len(self.colors)
            self.canvas.itemconfig(self.rect, fill=self.colors[self.current_color])
        # Reset position when 'r' is pressed
        elif key.lower() == "r":
            self.canvas.coords(self.rect, 180, 80, 220, 120)
            self.canvas.itemconfig(self.rect, fill="blue")
            self.current_color = 0

if __name__ == "__main__":
    root = tk.Tk()
    app = KeyboardControlApp(root)
    root.mainloop()